In [ ]:
import optuna
import xgboost as xgb
import polars as pl
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import gc
# --- 1. Configuration & Data Loading ---
k_clusters = 512                            # 512 is usually the sweetspot
model_name = "qwen3-8b-with-instruction"    # qwen3-with-instruction is best

# --- 1. Load & Shuffle ---
dat = pl.read_parquet(f"../../../data/clustered_embeddings/{model_name}/autoencoded_clusters_{k_clusters}.parquet")
dat = dat.sample(fraction=1.0, seed=42).filter((pl.col("r").is_not_null()) & (pl.col("r") != 1))

# --- 2. Sacred Holdout (10%) ---
sacred_holdout_df = dat.sample(fraction=0.1, seed=99)
work_df = dat.join(sacred_holdout_df, on=dat.columns, how="anti")

del dat
gc.collect()

# Identify columns
nominal_cols = ["top_sentiment_item1", "top_emotion_item1", "top_sentiment_item2", "top_emotion_item2"]
numeric_cols = [c for c in work_df.columns if work_df[c].dtype.is_numeric() and not c == 'r']

def objective(trial):
    param = {
        'n_estimators': 1000,
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda:0',
        'learning_rate': trial.suggest_float("learning_rate", 0.001, 0.2, log=True),
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 10),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'gamma': trial.suggest_float("gamma", 0.0, 5.0),
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_corrs = []

    # Iterate through folds
    for train_idx, val_idx in kf.split(work_df):
        # 1. Split RAW data
        train_fold = work_df.slice(train_idx[0], len(train_idx)) # polars slice
        val_fold = work_df.slice(val_idx[0], len(val_idx))

        # 2. FIT SCALER/ENCODER ONLY ON TRAIN FOLD
        scaler = StandardScaler()
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        
        # Numeric
        X_train_num = scaler.fit_transform(train_fold.select(numeric_cols).to_numpy()).astype(np.float32)
        X_val_num = scaler.transform(val_fold.select(numeric_cols).to_numpy()).astype(np.float32)
        
        # Categorical
        X_train_cat = encoder.fit_transform(train_fold.select(nominal_cols).to_numpy())
        X_val_cat = encoder.transform(val_fold.select(nominal_cols).to_numpy())
        
        # Combine
        X_train = np.hstack([X_train_num, X_train_cat])
        X_val = np.hstack([X_val_num, X_val_cat])
        
        y_train = train_fold.select("r").to_numpy().flatten().astype(np.float32)
        y_val = val_fold.select("r").to_numpy().flatten().astype(np.float32)

        
        # 3. Train
        model = xgb.XGBRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        # 4. Score
        preds = model.predict(X_val)
        corr = np.corrcoef(y_val, preds)[0, 1]
        fold_corrs.append(corr if not np.isnan(corr) else -1.0)

    return np.mean(fold_corrs)

# --- Tuning ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# --- Final Check on Sacred Holdout ---
# Repeat the logic: Fit on ALL work_df, transform sacred_holdout_df
scaler_final = StandardScaler()
encoder_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_work_num = scaler_final.fit_transform(work_df.select(numeric_cols).to_numpy())
X_sacred_num = scaler_final.transform(sacred_holdout_df.select(numeric_cols).to_numpy())

X_work_cat = encoder_final.fit_transform(work_df.select(nominal_cols).to_numpy())
X_sacred_cat = encoder_final.transform(sacred_holdout_df.select(nominal_cols).to_numpy())

X_work_final = np.hstack([X_work_num, X_work_cat])
X_sacred_final = np.hstack([X_sacred_num, X_sacred_cat])
y_work_final = work_df.select("r").to_numpy().flatten()
y_sacred_final = sacred_holdout_df.select("r").to_numpy().flatten()

best_model = xgb.XGBRegressor(**study.best_trial.params, device='cuda:0')
best_model.fit(X_work_final, y_work_final, verbose=False)
final_corr = np.corrcoef(y_sacred_final, best_model.predict(X_sacred_final))[0, 1]

print(f"Final Sacred Holdout Correlation: {final_corr:.4f}")

[I 2026-04-11 22:55:15,175] A new study created in memory with name: no-name-704e1569-77a0-4ad7-9449-ad524c9f209d
[I 2026-04-11 22:55:43,891] Trial 0 finished with value: 0.6944938752301063 and parameters: {'learning_rate': 0.1154235297849069, 'max_depth': 8, 'min_child_weight': 7, 'subsample': 0.8429792245012748, 'gamma': 3.205688162464824}. Best is trial 0 with value: 0.6944938752301063.
[I 2026-04-11 22:56:10,408] Trial 1 finished with value: 0.663866880152531 and parameters: {'learning_rate': 0.056866320231945565, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.7101790959359406, 'gamma': 4.260686511372899}. Best is trial 0 with value: 0.6944938752301063.
[I 2026-04-11 22:57:10,884] Trial 2 finished with value: 0.8484298801863073 and parameters: {'learning_rate': 0.01742169156688835, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.5980627302191013, 'gamma': 0.0631931060125851}. Best is trial 2 with value: 0.8484298801863073.
[I 2026-04-11 22:57:44,802] Trial 3 finished 

Final Sacred Holdout Correlation: 0.8093


Building on these trials we initiate another trial round using more strictly defined ranges for the hyperparameters.


In [6]:
import optuna
import xgboost as xgb
import polars as pl
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import gc
# --- 1. Configuration & Data Loading ---
k_clusters = 512                            # 512 is usually the sweetspot
model_name = "qwen3-8b-with-instruction"    # qwen3-with-instruction is best

# --- 1. Load & Shuffle ---
dat = pl.read_parquet(f"../../../data/clustered_embeddings/{model_name}/autoencoded_clusters_{k_clusters}.parquet")
dat = dat.sample(fraction=1.0, seed=42).filter((pl.col("r").is_not_null()) & (pl.col("r") != 1))

# --- 2. Sacred Holdout (10%) ---
sacred_holdout_df = dat.sample(fraction=0.1, seed=99)
work_df = dat.join(sacred_holdout_df, on=dat.columns, how="anti")

del dat
gc.collect()

# Identify columns
nominal_cols = ["top_sentiment_item1", "top_emotion_item1", "top_sentiment_item2", "top_emotion_item2"]
numeric_cols = [c for c in work_df.columns if work_df[c].dtype.is_numeric() and not c == 'r']

def objective(trial):
    param = {
        'n_estimators': 1000,
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda:0',
        'learning_rate': trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        'max_depth': trial.suggest_int("max_depth", 3, 7),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 9),
        'subsample': trial.suggest_float("subsample", 0.5, 0.8),
        'gamma': trial.suggest_float("gamma", 0.0, 0.8),
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_corrs = []
    fold_rmses = []

    # Iterate through folds
    for train_idx, val_idx in kf.split(work_df):
        # 1. Split RAW data
        train_fold = work_df.slice(train_idx[0], len(train_idx)) # polars slice
        val_fold = work_df.slice(val_idx[0], len(val_idx))

        # 2. FIT SCALER/ENCODER ONLY ON TRAIN FOLD
        scaler = StandardScaler()
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        
        # Numeric
        X_train_num = scaler.fit_transform(train_fold.select(numeric_cols).to_numpy()).astype(np.float32)
        X_val_num = scaler.transform(val_fold.select(numeric_cols).to_numpy()).astype(np.float32)
        
        # Categorical
        X_train_cat = encoder.fit_transform(train_fold.select(nominal_cols).to_numpy())
        X_val_cat = encoder.transform(val_fold.select(nominal_cols).to_numpy())
        
        # Combine
        X_train = np.hstack([X_train_num, X_train_cat])
        X_val = np.hstack([X_val_num, X_val_cat])
        
        y_train = train_fold.select("r").to_numpy().flatten().astype(np.float32)
        y_val = val_fold.select("r").to_numpy().flatten().astype(np.float32)

        
        # 3. Train
        model = xgb.XGBRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        # 4. Score
        preds = model.predict(X_val)
        corr = np.corrcoef(y_val, preds)[0, 1]
        fold_corrs.append(corr if not np.isnan(corr) else -1.0)

        rmse = np.sqrt(np.mean((y_val - preds)**2))
        fold_rmses.append(rmse)

    return np.mean(fold_corrs), np.mean(fold_rmses)

# --- Tuning ---
study = optuna.create_study(directions=["maximize", "minimize"])
study.optimize(objective, n_trials=10)

print("-" * 30)
print("Pareto Front (Best Trade-off Trials):")
for trial in study.best_trials:
    print(f"  Trial {trial.number}: Corr={trial.values[0]:.4f}, RMSE={trial.values[1]:.4f}")
    print(f"  Params: {trial.params}")

# --- Final Check on Sacred Holdout ---
# Repeat the logic: Fit on ALL work_df, transform sacred_holdout_df
best_trial = max(study.best_trials, key=lambda t: t.values[0])
scaler_final = StandardScaler()
encoder_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_work_num = scaler_final.fit_transform(work_df.select(numeric_cols).to_numpy())
X_sacred_num = scaler_final.transform(sacred_holdout_df.select(numeric_cols).to_numpy())

X_work_cat = encoder_final.fit_transform(work_df.select(nominal_cols).to_numpy())
X_sacred_cat = encoder_final.transform(sacred_holdout_df.select(nominal_cols).to_numpy())

X_work_final = np.hstack([X_work_num, X_work_cat])
X_sacred_final = np.hstack([X_sacred_num, X_sacred_cat])
y_work_final = work_df.select("r").to_numpy().flatten()
y_sacred_final = sacred_holdout_df.select("r").to_numpy().flatten()

best_model = xgb.XGBRegressor(**best_trial.params, n_estimators=1000, device='cuda:0')
best_model.fit(X_work_final, y_work_final, verbose=False)


final_preds = best_model.predict(X_sacred_final)
final_corr = np.corrcoef(y_sacred_final, final_preds)[0, 1]
final_rmse = np.sqrt(np.mean((y_sacred_final - final_preds)**2))

print("\n[Final Sacred Holdout Results]")
print(f"Correlation: {final_corr:.4f}")
print(f"RMSE: {final_rmse:.4f}")

[I 2026-04-11 23:20:55,871] A new study created in memory with name: no-name-6108703d-2d3f-47c9-b467-15a8486306a0
[I 2026-04-11 23:21:38,501] Trial 0 finished with values: [0.7911804419889683, 0.10474326461553574] and parameters: {'learning_rate': 0.021204044674911042, 'max_depth': 6, 'min_child_weight': 4, 'subsample': 0.592077686698192, 'gamma': 0.5911196081542116}.
[I 2026-04-11 23:22:35,974] Trial 1 finished with values: [0.7762474016606592, 0.10787391662597656] and parameters: {'learning_rate': 0.0076668788628414765, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.5698617838341336, 'gamma': 0.5598681481124909}.
[I 2026-04-11 23:23:09,160] Trial 2 finished with values: [0.7770336205655441, 0.10732883960008621] and parameters: {'learning_rate': 0.12471829868933373, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.537431904037042, 'gamma': 0.7077773041978054}.
[I 2026-04-11 23:23:44,020] Trial 3 finished with values: [0.825843473196549, 0.09665048122406006] and parameters:

------------------------------
Pareto Front (Best Trade-off Trials):
  Trial 6: Corr=0.8826, RMSE=0.0811
  Params: {'learning_rate': 0.06596888389853432, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.7976895019891392, 'gamma': 0.1670321962980749}

[Final Sacred Holdout Results]
Correlation: 0.8624
RMSE: 0.0994
